<a href="https://www.kaggle.com/code/urvishahir/bert-fine-tuning-for-sentiment-analysis-imdb?scriptVersionId=310409214" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# BERT Fine Tuning for Sentiment Analysis (IMDB)

## Objective
The goal of this notebook is to fine-tune a pre-trained BERT model for sentiment classification using the IMDB movie reviews dataset.

## Concepts Covered
- Text preprocessing for NLP
- Tokenization using BERT
- Fine-tuning transformer models
- Model evaluation using classification metrics
- Experimental comparison of different training strategies

## Dataset
The dataset consists of 50,000 movie reviews labeled as positive or negative.

## Workflow
Raw Data > Preprocessing > Tokenization > Dataset > Model Training > Evaluation > Experiments

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm

---

## Load Dataset

We load the IMDB dataset which contains labeled movie reviews for sentiment analysis.

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.head()
df.head()

---

## Data Exploration

Understanding the structure of the dataset and checking class distribution.

In [ ]:
df.info()

In [ ]:
df['sentiment'].value_counts()

---

## Data Preprocessing

Before training the model, we perform basic preprocessing. Since BERT is a contextual model, only minimal cleaning is applied.

Steps performed:
- Handle missing values
- Convert labels to numeric format
- Clean text by removing unwanted elements

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

Handling missing values ensures that the dataset does not contain null entries, which could cause issues during training.

---

## Convert Labels to Numeric Format

Machine learning models require numeric labels. We convert:
- positive -> 1
- negative -> 0

In [ ]:
df['sentiment'].unique()

In [ ]:
df['sentiment'] = df['sentiment'].str.strip().str.lower()

Before label conversion, we normalize the sentiment column by removing extra spaces and converting text to lowercase. This ensures correct mapping to numeric labels.

In [ ]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [ ]:
df.head()

This step converts categorical labels into numeric values so that the model can process them during training.

---

## Text Cleaning

We perform minimal preprocessing to remove noise from the text while preserving its meaning.

Steps:
- Convert text to lowercase
- Remove HTML tags
- Remove special characters

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['review'] = df['review'].apply(clean_text)

In [ ]:
df.head()

Text is cleaned to remove unnecessary elements such as HTML tags and special characters. 
Only basic cleaning is performed because BERT can understand context and language structure effectively.

---

## Data Splitting

The dataset is split into training, validation, and test sets.

- Training set is used to train the model
- Validation set is used to tune the model
- Test set is used for final evaluation

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Train (70%) and Temp (30%)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.3,
    random_state=42
)

# Step 2: Split Temp into Validation (15%) and Test (15%)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42
)

In [ ]:
print(len(train_texts))
print(len(val_texts))
print(len(test_texts))

The dataset is first split into training and temporary sets (70% and 30%).  
The temporary set is further divided equally into validation and test sets.

This ensures proper evaluation and prevents data leakage.

---

## Tokenization

We use the pre-trained BERT tokenizer to convert text into token IDs.  
This step prepares the input data in a format suitable for the BERT model.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

To monitor progress and improve usability, we process the data in batches and use a progress bar.

This helps track how much of the data has been processed during tokenization.

In [ ]:
def tokenize_in_batches(texts, tokenizer, batch_size=1000):
    encodings = {'input_ids': [], 'attention_mask': []}
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        tokenized = tokenizer(batch, truncation=True, padding=True)
        
        encodings['input_ids'].extend(tokenized['input_ids'])
        encodings['attention_mask'].extend(tokenized['attention_mask'])
    
    return encodings

In [ ]:
print("Tokenizing training data...")
train_encodings = tokenize_in_batches(list(train_texts), tokenizer)

print("Tokenizing validation data...")
val_encodings = tokenize_in_batches(list(val_texts), tokenizer)

print("Tokenizing test data...")
test_encodings = tokenize_in_batches(list(test_texts), tokenizer)

In [ ]:
train_encodings.keys()

The tokenizer converts text into token IDs and generates attention masks.  
These are required inputs for the BERT model.

---

## Creating Dataset

After tokenization, the data is converted into a custom PyTorch dataset.

This step is required because the Trainer expects inputs in a specific format that includes:
- input_ids
- attention_mask
- labels

The dataset class helps organize tokenized data and labels so they can be used during training.

In [ ]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)

---

## Load BERT Model

We load a pre-trained BERT model for sequence classification.

The model already understands language patterns and is fine-tuned on our dataset.

We set `num_labels=2` since this is a binary classification problem (positive and negative).

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

---

## Training Configuration

We define training parameters such as learning rate, batch size, and number of epochs. These control how the model learns during fine-tuning.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2
)

---

## Trainer Setup

The Trainer API simplifies the training process by handling batching, evaluation, and optimization.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

---

## Model Training

We start fine-tuning the BERT model on the training dataset.

In [ ]:
trainer.train()

---

## Evaluation

After training, the model is evaluated on the test dataset.

We use standard classification metrics:
- Accuracy
- Precision
- Recall
- F1 Score

A confusion matrix is also used to visualize prediction performance.

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(test_labels, preds))
print(confusion_matrix(test_labels, preds))

---

## Analysis

The BERT model achieved an accuracy of 94%, indicating strong performance on the IMDB dataset.

The model shows balanced precision and recall across both classes, meaning it does not favor positive or negative predictions.

The confusion matrix shows a low number of misclassifications, suggesting that the model generalizes well.

This performance highlights the effectiveness of transformer-based models like BERT in understanding contextual language patterns compared to traditional NLP methods.

---

## Experiment 1: Freeze BERT Layers

In this experiment, we freeze all BERT layers and train only the classification head.

In [ ]:
from transformers import AutoModelForSequenceClassification

model_frozen = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Freeze all BERT layers
for param in model_frozen.bert.parameters():
    param.requires_grad = False

In [ ]:
# Training Setup
from transformers import TrainingArguments

training_args_exp = TrainingArguments(
    output_dir='./results_exp1',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)

In [ ]:
# Trainer
trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args_exp,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer_frozen.train()

In [ ]:
# Evaluation
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer_frozen.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, preds))
print(confusion_matrix(test_labels, preds))

## Experiment 1 Analysis

The frozen BERT model achieved an accuracy of 60%, which is significantly lower than the fully fine-tuned model (94%).

This is because all BERT layers were frozen, preventing the model from learning dataset-specific patterns.

The confusion matrix shows a high number of misclassifications, especially for the negative class, indicating poor adaptability.

This demonstrates that fine-tuning BERT is essential for achieving high performance.

---

## Experiment 2: Fine-Tuning Last 2 Layers

In this experiment, we freeze all BERT layers except the last two layers. This allows partial learning while reducing training complexity.

In [ ]:
model_partial = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

for param in model_partial.bert.parameters():
    param.requires_grad = False

for name, param in model_partial.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True

In [ ]:
# Training Setup
training_args_exp2 = TrainingArguments(
    output_dir='./results_exp2',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)

In [ ]:
# Trainer
trainer_partial = Trainer(
    model=model_partial,
    args=training_args_exp2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer_partial.train()

In [ ]:
# Evaluation
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer_partial.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, preds))
print(confusion_matrix(test_labels, preds))

---

## Experiment 2 Analysis

The model achieved an accuracy of 91%, which is significantly better than the frozen BERT model.

By unfreezing only the last two layers, the model was able to learn task-specific patterns while keeping most of the pre-trained knowledge intact.

This approach provides a good balance between performance and training efficiency.

---

## Model Comparison

We compute accuracy and F1 score for all models and compare their performance.

The results are displayed in a table and visualized using a bar chart for better understanding.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def get_metrics(trainer, dataset, true_labels):
    predictions = trainer.predict(dataset)
    preds = predictions.predictions.argmax(axis=1)
    
    acc = accuracy_score(true_labels, preds)
    f1 = f1_score(true_labels, preds)
    
    return acc, f1

In [ ]:
acc_main, f1_main = get_metrics(trainer, test_dataset, test_labels)
acc_frozen, f1_frozen = get_metrics(trainer_frozen, test_dataset, test_labels)
acc_partial, f1_partial = get_metrics(trainer_partial, test_dataset, test_labels)

comparison = pd.DataFrame({
    "Model": ["Full Fine-Tuning", "Frozen BERT", "Last 2 Layers"],
    "Accuracy": [acc_main, acc_frozen, acc_partial],
    "F1 Score": [f1_main, f1_frozen, f1_partial]
})

comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.set_index("Model")["Accuracy"].plot(kind="bar")
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()

---

## Final Analysis

The fully fine-tuned BERT model achieved the highest accuracy of 94%, showing the best performance.

Freezing all BERT layers reduced accuracy to 60%, as the model could not adapt to the dataset.

Fine-tuning only the last two layers achieved 91% accuracy, demonstrating that partial fine-tuning can still produce strong results.

Overall, full fine-tuning provides the best performance, while partial fine-tuning offers a good trade-off between accuracy and computational cost.

---

## Conclusion

This project demonstrates how BERT can be fine-tuned for sentiment analysis. The results show that full fine-tuning achieves the best performance, while partial fine-tuning provides a good balance between accuracy and efficiency.

---